# 03 - Model training and validation

Compared six candidate configurations, tuned only on validation patients, and refitted frozen final pipelines without accessing held-out test outcomes.

## 1. Imports and preprocessed inputs

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").is_dir():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "processed"
MODEL_DIR = DATA_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
feature_files = {"Mutation": "mutation_feature_matrix.csv",
                 "CNA": "cna_feature_matrix_common.csv",
                 "Combined": "combined_feature_matrix.csv"}
feature_sets = {name: pd.read_csv(DATA_DIR / filename, index_col="PATIENT_ID")
                for name, filename in feature_files.items()}
splits = pd.read_csv(DATA_DIR / "patient_splits.csv")
print({name: x.shape for name, x in feature_sets.items()})

## 2. Validate alignment and freeze split assignments

In [ ]:
assert {"PATIENT_ID", "stage_binary", "split"}.issubset(splits.columns)
assert splits["PATIENT_ID"].is_unique
assert set(splits["split"]) == {"train", "validation", "test"}
train_ids = splits.loc[splits["split"] == "train", "PATIENT_ID"].tolist()
val_ids = splits.loc[splits["split"] == "validation", "PATIENT_ID"].tolist()
test_ids = splits.loc[splits["split"] == "test", "PATIENT_ID"].tolist()
assert len(set(train_ids) & set(val_ids)) == 0
assert len(set(train_ids) & set(test_ids)) == 0
assert len(set(val_ids) & set(test_ids)) == 0
y = splits.set_index("PATIENT_ID")["stage_binary"]
y_train, y_val = y.loc[train_ids], y.loc[val_ids]
for name, X in feature_sets.items():
    assert set(X.index) == set(y.index), name
    assert X.index.is_unique and X.columns.is_unique, name
    assert np.isfinite(X.to_numpy(dtype=float)).all(), name
print(f"Train: {len(train_ids)}; validation: {len(val_ids)}; held-out test: {len(test_ids)}")

## 3. Define reproducible model pipelines


In [ ]:
def build_model(model_name):
    steps = [("nonconstant", VarianceThreshold(threshold=0.0))]
    if model_name == "Logistic Regression":
        steps.extend([
            ("scale", StandardScaler()),
            ("classifier", LogisticRegression(C=1.0, max_iter=3000,
                                               solver="liblinear", random_state=RANDOM_STATE)),
        ])
    elif model_name == "Random Forest":
        steps.append(("classifier", RandomForestClassifier(n_estimators=300,
                                   min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)))
    else:
        raise ValueError(model_name)
    return Pipeline(steps)

model_names = ("Logistic Regression", "Random Forest")

## 4. Baseline model training and validation

In [ ]:
baseline_rows = []
for feature_name, X in feature_sets.items():
    for model_name in model_names:
        pipeline = build_model(model_name)
        pipeline.fit(X.loc[train_ids], y_train)
        p = pipeline.predict_proba(X.loc[val_ids])[:, 1]
        baseline_rows.append({"Feature Set": feature_name, "Model": model_name,
                              "Features Used": int(pipeline.named_steps["nonconstant"].get_support().sum()),
                              "Validation AUROC": roc_auc_score(y_val, p),
                              "Validation AP": average_precision_score(y_val, p)})
baseline_df = pd.DataFrame(baseline_rows).sort_values("Validation AUROC", ascending=False)
baseline_df.to_csv(DATA_DIR / "validation_model_comparison.csv", index=False)
display(baseline_df.round(3))

## 5. Prespecified hyperparameter search

Used the same validation patients for all comparisons. Test set untouched during model selection.

In [ ]:
tuning_rows = []
for feature_name, X in feature_sets.items():
    for C_value in [0.01, 0.1, 1.0, 10.0]:
        params = {"classifier__C": C_value}
        pipeline = build_model("Logistic Regression").set_params(**params)
        pipeline.fit(X.loc[train_ids], y_train)
        p = pipeline.predict_proba(X.loc[val_ids])[:, 1]
        tuning_rows.append({"Feature Set": feature_name, "Model": "Logistic Regression",
                            "Parameters": f"C={C_value}", "params": params,
                            "AUROC": roc_auc_score(y_val, p), "AP": average_precision_score(y_val, p)})
    for depth, leaf in [(None, 2), (None, 5), (10, 5), (10, 10)]:
        params = {"classifier__max_depth": depth, "classifier__min_samples_leaf": leaf}
        pipeline = build_model("Random Forest").set_params(**params)
        pipeline.fit(X.loc[train_ids], y_train)
        p = pipeline.predict_proba(X.loc[val_ids])[:, 1]
        tuning_rows.append({"Feature Set": feature_name, "Model": "Random Forest",
                            "Parameters": f"depth={depth}, leaf={leaf}", "params": params,
                            "AUROC": roc_auc_score(y_val, p), "AP": average_precision_score(y_val, p)})
tuning_df = pd.DataFrame(tuning_rows)
best_tuning = (tuning_df.sort_values(["AUROC", "AP"], ascending=False)
               .drop_duplicates(["Feature Set", "Model"]).reset_index(drop=True))
tuning_df.drop(columns="params").to_csv(DATA_DIR / "hyperparameter_tuning.csv", index=False)
display(best_tuning.drop(columns="params").round(3))

## 6. Refit and save finalized models

After validation-only hyperparameter selection, refitted using training + validation.

In [ ]:
train_val_ids = splits.loc[splits["split"].isin(["train", "validation"]), "PATIENT_ID"].tolist()
assert set(train_val_ids).isdisjoint(test_ids)
assert len(train_val_ids) == len(train_ids) + len(val_ids)
for row in best_tuning.itertuples(index=False):
    feature_name = row[0]
    model_name = row[1]
    params = row[3]  # Column `params`; stored as a Python dict (not the human-readable string).
    model = build_model(model_name).set_params(**params)
    model.fit(feature_sets[feature_name].loc[train_val_ids], y.loc[train_val_ids])
    filename = f"{feature_name.lower()}_{model_name.lower().replace(' ', '_')}.joblib"
    joblib.dump(model, MODEL_DIR / filename)
    print("Saved:", filename)
print("PASS: Six frozen models saved; held-out test not evaluated.")